# NCAA Hockey Play-by-Play Data Scraper
## Updated for 2025-26 Season
- necessary because of update to html structure of 

In [10]:
# NCAA Hockey Play-by-Play → tidy DataFrame for pages like:
# https://www.ncaa.com/game/6498822/play-by-play
# Output columns: Period, Time, Team, Play, Score

from __future__ import annotations
import re, asyncio
from typing import List, Dict, Optional
import pandas as pd
from playwright.async_api import async_playwright

# ---------- core scraper (async, Jupyter-safe) ----------
async def _scrape_pbp_table(url: str) -> pd.DataFrame:
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        context = await browser.new_context(
            user_agent=(
                "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                "AppleWebKit/537.36 (KHTML, like Gecko) "
                "Chrome/123.0.0.0 Safari/537.36"
            ),
            locale="en-US",
        )

        # Block non-essential resources to speed up
        async def route_block(route, request):
            rtype = request.resource_type
            if rtype in ("image","font","stylesheet","media"):
                return await route.abort()
            u = request.url.lower()
            if any(s in u for s in ("googletagmanager","doubleclick","analytics","scorestream")):
                return await route.abort()
            return await route.continue_()
        await context.route("**/*", route_block)

        page = await context.new_page()
        await page.goto(url, wait_until="domcontentloaded", timeout=25000)

        # Wait for any PBP table to appear
        await page.wait_for_selector("section.play-by-play table, div.play-by-play table, #play-by-play table, table", timeout=15000)

        # Extract rows in-page to avoid fragile re-parsing
        js = """
(() => {
  function txt(el){ return (el && el.textContent ? el.textContent : "").replace(/\\s+/g," ").trim(); }
  function normPeriod(t){
    t = String(t||"").trim().toUpperCase();
    const map = {FIRST:"1",SECOND:"2",THIRD:"3","1ST":"1","2ND":"2","3RD":"3",OVERTIME:"OT",OT:"OT"};
    if (map[t]) return map[t];
    const m = t.match(/^(\\d+)\\s*OT$/); if (m) return m[1] + "OT";
    const d = t.match(/\\b([123])\\b/); return d ? d[1] : t || "";
  }
  function headingBefore(node, root){
    let cur = node;
    while (cur && cur !== root) {
      // climb siblings first
      let p = cur.previousElementSibling;
      while (p) {
        if (/^H[2-4]$/.test(p.tagName)) return txt(p);
        p = p.previousElementSibling;
      }
      cur = cur.parentElement;
    }
    return "";
  }

  const root = document.querySelector("section.play-by-play") ||
               document.querySelector("div.play-by-play") ||
               document.querySelector("#play-by-play") || document;

  const tables = Array.from(root.querySelectorAll("table"));
  const out = [];

  for (const tbl of tables) {
    const periodRaw = headingBefore(tbl, root);
    const period = normPeriod(periodRaw);

    const thead = tbl.querySelector("thead");
    const heads = thead ? Array.from(thead.querySelectorAll("th")).map(h => txt(h).toLowerCase()) : [];

    // header → index
    const idxTime  = heads.findIndex(h => /time/.test(h));
    const idxTeam  = heads.findIndex(h => /team/.test(h));
    const idxPlay  = heads.findIndex(h => /play/.test(h));
    const idxScore = heads.findIndex(h => /score/.test(h));

    const trs = Array.from(tbl.querySelectorAll("tbody tr"));
    for (const tr of trs) {
      const cells = Array.from(tr.querySelectorAll("td")).map(td => txt(td));
      if (!cells.length) continue;

      // fallback positions if headers absent (rare)
      const tTime  = idxTime  >= 0 ? cells[idxTime]  : (cells[0] || "");
      const tTeam  = idxTeam  >= 0 ? cells[idxTeam]  : (cells.length >= 4 ? cells[1] : "");
      const tPlay  = idxPlay  >= 0 ? cells[idxPlay]  : (cells.length >= 4 ? cells[2] : (cells[1] || ""));
      const tScore = idxScore >= 0 ? cells[idxScore] : (cells.length >= 3 ? cells[cells.length-1] : "");

      let Team = tTeam;
      let Play = tPlay;

      // If "Team: Play ..." pattern, peel team from Play
      if (!Team && /^[A-Za-z .&'()-]+:\\s+/.test(Play)) {
        const m = Play.match(/^([A-Za-z .&'()-]+):\\s*(.*)$/);
        if (m) { Team = m[1]; Play = m[2]; }
      }

      // Drop junk rows
      if (!Play || Play.length < 2) continue;

      out.push({
        Period: period || "",
        Time: tTime || "",
        Team: Team || "",
        Play: Play || "",
        Score: (tScore || "").replace(/\\s+/g,"")
      });
    }
  }

  return out;
})();
"""
        rows = await page.evaluate(js)
        await browser.close()

    df = pd.DataFrame(rows, columns=["Period","Time","Team","Play","Score"])
    # Basic tidy
    for col in ["Period","Time","Team","Play","Score"]:
        df[col] = df[col].fillna("").astype(str).str.strip()

    # Heuristic: if some Periods are blank, infer from "Start of X Period" markers & forward-fill
    if df["Period"].eq("").any():
        def infer_from_play(s: str) -> Optional[str]:
            s2 = s.upper()
            m = re.search(r"START OF\\s*(FIRST|SECOND|THIRD|1ST|2ND|3RD|OT|\\d+OT)\\s*PERIOD", s2)
            if not m: 
                # some pages omit "PERIOD"
                m = re.search(r"START OF\\s*(FIRST|SECOND|THIRD|1ST|2ND|3RD|OT|\\d+OT)", s2)
            if not m: 
                return None
            tok = m.group(1)
            mapping = {"FIRST":"1","SECOND":"2","THIRD":"3","1ST":"1","2ND":"2","3RD":"3"}
            return mapping.get(tok, tok)
        inferred = df["Play"].map(infer_from_play)
        cur = ""
        filled = []
        for p, inf in zip(df["Period"], inferred):
            if p: cur = p
            elif inf: cur = inf
            filled.append(cur)
        df["Period"] = filled

    # Order period categories if possible
    cat_order = ["1","2","3","OT","2OT","3OT"]
    try:
        df["Period"] = pd.Categorical(df["Period"].replace({"FIRST":"1","SECOND":"2","THIRD":"3"}), cat_order, ordered=True)
    except Exception:
        pass

    # Final filter: keep only rows where Play is present
    df = df[df["Play"].ne("")].reset_index(drop=True)
    # Rename to target column names
    df = df.rename(columns={"Team":"team","Play":"play","Score":"score","Time":"time","Period":"Period"})
    # Reorder exactly as you want (Period, team, play, score) — include time if you want it too
    df = df[["Period","time","team","play","score"]]
    return df

# ---------- convenience wrapper ----------
async def get_game_pbp_df(url_or_id: str|int) -> pd.DataFrame:
    import re
    s = str(url_or_id)
    if re.search(r"^https?://", s) and "/game/" in s:
        url = s if s.endswith("/play-by-play") else (s.rstrip("/") + "/play-by-play")
    else:
        url = f"https://www.ncaa.com/game/{s}/play-by-play"
    return await _scrape_pbp_table(url)

# ---------- run on your example ----------
url = "https://www.ncaa.com/game/6498822/play-by-play"
df = await get_game_pbp_df(url)
print(df.head(15))
print("Shape:", df.shape)
# df.to_csv("pbp_6498822.csv", index=False)


   Period   time team                                               play score
0     NaN  20:00                        Melvin Strahl at goalie for MSU.      
1     NaN  20:00       Faceoff Silkalns, Girts vs Stramel, Charlie wo...      
2     NaN  20:00                Oliver Auyeung-Ashton at goalie for NMU.      
3     NaN  19:28       Shot by NMU Silkalns, Girts BLOCKED by Stramel...      
4     NaN  19:28       Faceoff Altrichter, Jakub vs Lindstrom, Cayden...      
5     NaN  19:28       Shot by NMU Gault, Caiden MISSED, save Strahl,...      
6     NaN  19:18       Shot by NMU Slipec, Grayden MISSED, save Strah...      
7     NaN  19:17       Faceoff Lindstrom, Cayden vs Altrichter, Jakub...      
8     NaN  18:54       Shot by MSU Basgall, Matt MISSED, save Auyeung...      
9     NaN  18:43       Shot by MSU Vansaghi, Shane MISSED, save Auyeu...      
10    NaN  18:37       Shot by MSU Ralph, Colin MISSED, save Auyeung-...      
11    NaN  18:29       Shot by MSU Vansaghi, Shane B

In [ ]:

# Optionally save
df.to_csv("pbp_raw.csv", index=False)


In [ ]:
df